In [ ]:
pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 34.8 MB/s eta 0:00:00


**SVM - Classifier**

In [ ]:
import os
import easyocr
import pandas as pd
import cv2
import joblib
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# ----------------------
# Initialize OCR
# ----------------------
reader = easyocr.Reader(['en'])

DATA_DIR = "/content/drive/MyDrive/project/dataset"

texts = []
labels = []

# OCR + cleaning
def extract_text(image_path):
    img = cv2.imread(image_path)

    # Resize for speed
    img = cv2.resize(img, (800, 800))

    result = reader.readtext(img, detail=0)
    text = " ".join(result).lower()

    # Clean text
    text = re.sub(r'[^a-zA-Z ]', '', text)

    return text

# ----------------------
# Load dataset
# ----------------------
for label in os.listdir(DATA_DIR):
    folder_path = os.path.join(DATA_DIR, label)

    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)

        print(f"Processing: {file_path}")

        try:
            text = extract_text(file_path)
            texts.append(text)
            labels.append(label)
        except Exception as e:
            print(f"Error: {file_path} → {e}")

# ----------------------
# Create DataFrame
# ----------------------
df = pd.DataFrame({
    "text": texts,
    "label": labels
})

print("\nTotal samples:", len(df))

# ----------------------
# Train-test split
# ----------------------
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.3,
    stratify=df["label"],
    random_state=42
)

# ----------------------
# TF-IDF vectorizer
# ----------------------
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ----------------------
# Model: Linear SVM
# ----------------------
model = LinearSVC(C=1.0, random_state=42, max_iter=5000)
model.fit(X_train_tfidf, y_train)

# ----------------------
# Predictions
# ----------------------
train_pred = model.predict(X_train_tfidf)
test_pred = model.predict(X_test_tfidf)

# ----------------------
# Accuracy
# ----------------------
train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

print("\n📊 Train Accuracy:", train_acc)
print("📊 Test Accuracy:", test_acc)

# ----------------------
# Classification report
# ----------------------
print("\n📄 Classification Report (Test Data):")
print(classification_report(y_test, test_pred))

# ----------------------
# Save model and vectorizer
# ----------------------
joblib.dump(model, "/content/drive/MyDrive/project/Source_code/model.pkl")
joblib.dump(vectorizer, "/content/drive/MyDrive/project/Source_code/vectorizer.pkl")

print("\n✅ Training completed and Linear SVM model saved!")

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteProcessing: /content/drive/MyDrive/project/dataset/scientific/40033964-3964.jpg
Processing: /content/drive/MyDrive/project/dataset/scientific/40021754-1754.jpg
Processing: /content/drive/MyDrive/project/dataset/scientific/40037985-7989.jpg
Processing: /content/drive/MyDrive/project/dataset/scientific/40036154-6154.jpg
Processing: /content/drive/MyDrive/project/dataset/scientific/40024983-4986.jpg
Processing: /content/drive/MyDrive/project/dataset/scientific/50455681-5681.jpg
Processing: /content/drive/MyDrive/project/dataset/scientific/50391006-1006.jpg
Processing: /content/drive/MyDrive/project/dataset/scientific/50473051-3053.jpg
Processing: /content/drive/MyDrive/project/dataset/scientific/50454742-4743.jpg
Processing: /content/drive/MyDrive/project/dataset/scientific/50380095-0099.jpg
Processing: /content/drive/MyDrive/project/dataset/scientific/50355312-5312.jpg
Processing: /content/drive/MyDrive/project